# Chatbot Inference Pipeline
Test the trained Municipality of Sampaloc Quezon Chatbot

In [1]:
import json
import pickle
import torch
import faiss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, T5TokenizerFast, T5ForConditionalGeneration
from sentence_transformers import SentenceTransformer

c:\Users\eyouth\Desktop\PrimeHrProjectMagdalena\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load All Models

In [2]:
# Time Waster Detection
with open('models/time_waster_tfidf.pkl', 'rb') as f:
    tfidf = pickle.load(f)
with open('models/time_waster_classifier.pkl', 'rb') as f:
    tw_classifier = pickle.load(f)

print("✓ Time waster detection loaded")

✓ Time waster detection loaded


In [3]:
# Sentiment Analysis
sentiment_tokenizer = AutoTokenizer.from_pretrained('models/sentiment_tokenizer')
sentiment_model = AutoModelForSequenceClassification.from_pretrained('models/sentiment_model')

print("✓ Sentiment analysis loaded")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 2758.85it/s]

✓ Sentiment analysis loaded


In [4]:
# RAG Retrieval
embedder = SentenceTransformer('all-MiniLM-L6-v2')
faiss_index = faiss.read_index('models/faiss_index.bin')

with open('models/documents.json', 'r', encoding='utf-8') as f:
    documents = json.load(f)

print("✓ RAG retrieval loaded")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3755.36it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ RAG retrieval loaded


In [5]:
# FLAN-T5
t5_tokenizer = T5TokenizerFast.from_pretrained('models/flan_t5_tokenizer')
t5_model = T5ForConditionalGeneration.from_pretrained('models/flan_t5_model')

print("✓ FLAN-T5 loaded")

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 2265.44it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✓ FLAN-T5 loaded


## Define Pipeline Functions

In [6]:
def preprocess(text):
    return text.strip().lower()

def detect_time_waster(text):
    vec = tfidf.transform([text])
    return tw_classifier.predict(vec)[0] == 1

def analyze_sentiment(text):
    inputs = sentiment_tokenizer(text, return_tensors="pt", truncation=True, max_length=512)
    outputs = sentiment_model(**inputs)
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    return "positive" if probs[0][1] > probs[0][0] else "negative"

def retrieve_context(query, k=3):
    query_embedding = embedder.encode([query])
    distances, indices = faiss_index.search(query_embedding.astype('float32'), k)
    return [documents[i] for i in indices[0]]

def generate_answer(query, context, sentiment):
    context_text = "\n".join([f"Service: {c['title']}. Office: {c['office']}" for c in context])
    
    if sentiment == "negative":
        tone = "Respond professionally with empathy and comfort. "
    else:
        tone = "Respond professionally and helpfully. "
    
    prompt = f"{tone}Context: {context_text}\n\nQuestion: {query}\n\nAnswer:"
    
    inputs = t5_tokenizer(prompt, return_tensors="pt", max_length=512, truncation=True)
    outputs = t5_model.generate(**inputs, max_length=150)
    return t5_tokenizer.decode(outputs[0], skip_special_tokens=True)

In [7]:
def process_query(user_input):
    # 1. Preprocessing
    text = preprocess(user_input)
    
    # 2. Time waster detection
    if detect_time_waster(text):
        return {"response": "Please provide a valid question about our services.", "status": "time_waster"}
    
    # 3. Sentiment analysis
    sentiment = analyze_sentiment(text)
    
    # 4. RAG retrieval
    context = retrieve_context(text)
    
    # 5. Generate answer
    answer = generate_answer(user_input, context, sentiment)
    
    # Add empathetic response for negative sentiment
    if sentiment == "negative":
        answer = f"I understand your concern. {answer} If you need further assistance, please feel free to ask."
    
    return {
        "response": answer,
        "sentiment": sentiment,
        "relevant_services": [c['title'] for c in context],
        "status": "success"
    }

## Test the Chatbot

In [8]:
# Test 1: Valid question
response = process_query("How do I get a business permit?")
print(json.dumps(response, indent=2))

KeyError: 'title'

In [ ]:
# Test 2: Time waster
response = process_query("hello")
print(json.dumps(response, indent=2))

{
  "response": "Please provide a valid question about our services.",
  "status": "time_waster"
}


In [ ]:
# Test 3: Negative sentiment
response = process_query("I'm frustrated! What are the requirements for marriage license?")
print(json.dumps(response, indent=2))

{
  "response": "I understand your concern. I'm frustrated! What are the requirements for marriage license? If you need further assistance, please feel free to ask.",
  "sentiment": "negative",
  "relevant_services": [
    "ISSUANCE OF PRINTED COPY OF LAND PARCEL OF THE SUBJECT PROPERTY",
    "PROVISION OF TOURISM ASSISTANCE",
    "PROVISION OF TOURISM ASSISTANCE"
  ],
  "status": "success"
}


In [ ]:
# Test 4: Your own question
user_question = "Do i need to pay in getting a barangay clearance?"
response = process_query(user_question)
print(json.dumps(response, indent=2))

{
  "response": "Please provide a valid question about our services.",
  "status": "time_waster"
}


## Interactive Testing

In [ ]:
while True:
    user_input = input("\nYou: ")
    if user_input.lower() in ['exit', 'quit']:
        break
    
    response = process_query(user_input)
    print(f"\nChatbot: {response['response']}")
    print(f"Sentiment: {response.get('sentiment', 'N/A')}")
    print(f"Status: {response['status']}")